In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV

# Pipeline
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

# Métricas de evaluación
from sklearn.metrics import accuracy_score


# Para guardar el modelo
import pickle

In [2]:
df = pd.read_csv('./data/titanic_procesado.csv')

# División de datos

In [3]:
X = df.drop(['Survived'], axis=1)
y = df['Survived']

X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1.0,1.0,0.433152,0.125,0.0,0.368146,1.0
1,0.0,0.0,0.579431,0.125,0.0,0.615097,0.0
2,1.0,0.0,0.462346,0.000,0.0,0.438286,1.0
3,0.0,0.0,0.563806,0.125,0.0,0.595112,1.0
4,1.0,1.0,0.563806,0.000,0.0,0.448347,1.0


In [5]:
y.head()

0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64

El test_size = 0.2 indica que el 20 % de los datos se van a test y el resto a entrenamiento.
Y el random_state = 42 es como una semilla de aleatoriedad, lo que permite repetir el mismo experimento con los mismos resultados; si no lo pusieramos, obtendríamos resultados diferentes todo el tiempo.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
X_train = X_train.values  # Convertir a NumPy array
y_train = y_train.values  # Convertir a NumPy array
X_test = X_test.values    # Convertir a NumPy array
y_test = y_test.values    # Convertir a NumPy array

# Modelos
Se pueden ver los hiperparámetros en:
https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

In [17]:
# Definir los modelos y sus respectivos hiperparámetros para GridSearch
modelos = {
    'Regresión Logística': {
        'modelo': LogisticRegression(),
        'parametros': {
            'C': [0.01, 0.1, 1, 10, 100],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear', 'saga'],
            'max_iter': [100, 500, 1000]
        }
    },
    'Clasificador de Vectores de Soporte': {
        'modelo': SVC(),
        'parametros': {
            'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
            'C': [0.1, 1, 10]
        }
    },
    'Clasificador de Árbol de Decisión': {
        'modelo': DecisionTreeClassifier(),
        'parametros': {
            'splitter': ['best', 'random'],
            'max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador de Bosques Aleatorios': {
        'modelo': RandomForestClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3, 4],
            'max_features': ['sqrt', 'log2', None]
        }
    },
    'Clasificador de Gradient Boosting': {
        'modelo': GradientBoostingClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador AdaBoost': {
        'modelo': AdaBoostClassifier(),
        'parametros': {
            'n_estimators': [10, 100]
        }
    },
    'Clasificador K-Nearest Neighbors': {
        'modelo': KNeighborsClassifier(),
        'parametros': {
            'n_neighbors': [3, 5, 7]
        }
    },
    'Clasificador XGBoost': {
        'modelo': XGBClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3]
        }
    },
    'Clasificador LGBM': {
        'modelo': LGBMClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3],
            'learning_rate': [0.1, 0.2, 0.3],
            'verbose': [-1]
        }
    },
    'GaussianNB': {
        'modelo': GaussianNB(),
        'parametros': {}
    },
    'Clasificador Naive Bayes': {
        'modelo': BernoulliNB(),
        'parametros': {
            'alpha': [0.1, 1.0, 10.0]
        }
    }
}

# Inicializar variables para almacenar los puntajes de los modelos y el mejor estimador
puntajes_modelos = []
mejor_precision = 0
mejor_estimador = None
mejor_modelo = None
estimadores = {}

# Iterar sobre cada modelo y sus hiperparámetros
for nombre, info_modelo in modelos.items():
    # Inicializar GridSearchCV con el modelo y los hiperparámetros
    grid_search = GridSearchCV(
        estimator=info_modelo['modelo'],
        param_grid=info_modelo['parametros'],
        cv=5,
        scoring='accuracy',
        verbose=0,
        n_jobs=10,
    )

    # Ajustar GridSearchCV con los datos de entrenamiento
    grid_search.fit(X_train, y_train)
    
    # Hacer predicciones con el modelo ajustado
    y_pred = grid_search.predict(X_test)
    
    # Calcular la precisión de las predicciones
    precision = accuracy_score(y_test, y_pred)
    
    # Almacenar los resultados del modelo
    puntajes_modelos.append({
        'Modelo': nombre,
        'Precisión': precision
    })

    estimadores[nombre] = grid_search.best_estimator_
    
    # Actualizar el mejor modelo si la precisión actual es mayor que la mejor precisión encontrada
    if precision > mejor_precision:
        mejor_modelo = nombre
        mejor_precision = precision
        mejor_estimador = grid_search.best_estimator_

# Convertir los resultados a un DataFrame para una mejor visualización
metricas = pd.DataFrame(puntajes_modelos).sort_values('Precisión', ascending=False)

# Imprimir el rendimiento de los modelos de clasificación
print("Rendimiento de los modelos de clasificación")
print(metricas.round(2))

# Imprimir el mejor modelo y su precisión
print('---------------------------------------------------')
print("MEJOR MODELO DE CLASIFICACIÓN")
print(f"Modelo: {mejor_modelo}")
print(f"Modelo: {mejor_estimador}")
print(f"Precisión: {mejor_precision:.2f}")

/home/edaxisrosnez/HYBRIDGE/Inteligencia_Artificial/proyecto2/aprendizaje_supervisado/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/edaxisrosnez/HYBRIDGE/Inteligencia_Artificial/proyecto2/aprendizaje_supervisado/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/home/edaxisrosnez/HYBRIDGE/Inteligencia_Artificial/proyecto2/aprendizaje_supervisado/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be remove

Rendimiento de los modelos de clasificación
                                 Modelo  Precisión
4     Clasificador de Gradient Boosting       0.84
6      Clasificador K-Nearest Neighbors       0.83
7                  Clasificador XGBoost       0.82
8                     Clasificador LGBM       0.81
3    Clasificador de Bosques Aleatorios       0.80
1   Clasificador de Vectores de Soporte       0.80
2     Clasificador de Árbol de Decisión       0.80
0                   Regresión Logística       0.79
5                 Clasificador AdaBoost       0.79
10             Clasificador Naive Bayes       0.79
9                            GaussianNB       0.76
---------------------------------------------------
MEJOR MODELO DE CLASIFICACIÓN
Modelo: Clasificador de Gradient Boosting
Modelo: GradientBoostingClassifier(max_depth=4)
Precisión: 0.84


In [19]:
list(modelos.keys())
mejor_modelo
mejor_estimador

,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are'friedman_mse' for the mean squared error with improvement score byFriedman, 'squared_error' for mean squared error. The default value of'friedman_mse' is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",4
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, 

# Entrenamiento de un solo modelo

## Sin hiperparámetros

In [14]:
# Esto ya lo tenemos importado. Lo ponemos nuevamente nada más de referencia
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# Creamos el modelo de regresión logística
model = LogisticRegression()

# Entrenamos el modelo con los datos de entrenamiento
model.fit(X_train, y_train)

# Realizamos predicciones con el conjunto de prueba
y_pred = model.predict(X_test)

# Evaluamos el modelo usando precisión
accuracy = accuracy_score(y_test, y_pred)

print(f"Precisión del modelo: {accuracy:.2f}")

Precisión del modelo: 0.80


## Con hiperparámetros

C: Controla la regularización del modelo. Un valor más bajo implica una regularización más fuerte (mayor penalización por pesos grandes), mientras que un valor más alto implica una regularización más débil. Por defecto es 1.0.

penalty: Especifica el tipo de penalización para la regularización. Los valores posibles son 'l1', 'l2', 'elasticnet', o None. 'l2' es el valor por defecto.

solver: El algoritmo utilizado para optimizar la función de coste. Los valores posibles incluyen 'newton-cg', 'lbfgs', 'liblinear', 'saga', y 'adam'. 'lbfgs' es el valor por defecto.

max_iter: El número máximo de iteraciones para el algoritmo de optimización. El valor por defecto es 100.

class_weight: Permite ajustar el peso de las clases en el entrenamiento. Puede ser None, 'balanced', o un diccionario con pesos específicos. Por defecto es None.

multi_class: Define el tipo de problema de clasificación múltiple. Los valores posibles son 'ovr' (one-vs-rest) y 'multinomial'. 'ovr' es el valor por defecto.

fit_intercept: Indica si se debe ajustar el intercepto. El valor por defecto es True.

warm_start: Permite reutilizar la solución de la llamada anterior para iniciar el siguiente ajuste. El valor por defecto es False.

tol: El umbral de tolerancia para el criterio de convergencia. El valor por defecto es 1e-4

In [15]:
model = LogisticRegression(
    C=0.5,                  # Valor de regularización
    penalty='l2',            # Tipo de penalización (l2 es la regularización Ridge)
    solver='lbfgs',         # Algoritmo de optimización
    max_iter=200,           # Número máximo de iteraciones
    class_weight='balanced' # Ajustar pesos de las clases
)

# Entrenar el modelo
model.fit(X_train, y_train)

# Realizar predicciones en el conjunto de prueba
y_pred = model.predict(X_test)

# Evaluar la precisión del modelo
accuracy = accuracy_score(y_test, y_pred)

print(f"Precisión del modelo: {accuracy:.2f}")

Precisión del modelo: 0.78


/home/edaxisrosnez/HYBRIDGE/Inteligencia_Artificial/proyecto2/aprendizaje_supervisado/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


# Inferencia

In [20]:
X_train[0]

array([0.        , 1.        , 0.61581699, 0.        , 0.        ,
       0.55559921, 1.        ])

In [21]:
y_train[0]

np.int64(0)

Prediciendo si un pasajero sobrevivió o no...

In [23]:
nuevos_datos = np.array([0,1,0.6159084,0,0,0.55547282,1]).reshape(1,-1)
mejor_estimador.predict(nuevos_datos)

array([0])

# Guardar el modelo

Pickle nos permite serializar (guardar) objetos de Python en un archivo para luego poder cargarlos y utilizarlos en diferentes entornos, como una API o una aplicación web.
Es particularmente útil cuando queremos guardar modelos entrenados o cualquier otro objeto complejo de Python. Al guardar el pipeline con Pickle, nos aseguramos de que todas las transformaciones de datos y el modelo en sí se conserven tal como fueron entrenados, permitiéndonos hacer predicciones consistentes con nuevos datos en producción sin necesidad de volver a aplicar las mismas transformaciones manualmente.

In [25]:
import pickle
with open('modelo.pkl', 'wb') as archivo_estimador:
    pickle.dump(mejor_estimador, archivo_estimador)  